# Домашнее задание 1 — банковские счета

Базовый класс `Account` и наследники `CheckingAccount`/`SavingsAccount`: учёт операций, валидация, аналитика и загрузка истории из «грязных» csv/json.

In [1]:
import csv
import json
import re
from dataclasses import dataclass, field
from datetime import UTC, datetime
from decimal import Decimal, InvalidOperation
from enum import StrEnum
from pathlib import Path


In [2]:
class OperationType(StrEnum):
    """Типы операций по счёту."""

    DEPOSIT = 'deposit'
    WITHDRAW = 'withdraw'
    INTEREST = 'interest'


class OperationStatus(StrEnum):
    """Статусы операций по счёту."""

    SUCCESS = 'success'
    FAIL = 'fail'


class AccountType(StrEnum):
    """Типы банковских счетов."""

    CHECKING = 'checking'
    SAVINGS = 'savings'


In [3]:
# Константы модуля.
HOLDER_PATTERN = re.compile(
    r'^[А-ЯЁA-Z][а-яёa-z]+ [А-ЯЁA-Z][а-яёa-z]+$'
)
OUTPUT_COLUMNS = ('type', 'amount', 'status', 'balance', 'date_time')
DATE_FORMATS = ('%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M')

In [4]:
# датакласс для хранения операции на счете
@dataclass
class Operation:
    """Структурированная запись об операции по счёту.

    Attributes:
        type: Тип операции.
        amount: Сумма операции.
        status: Статус операции.
        balance: Баланс после операции.
        date_time: Дата и время операции.
    """

    type: OperationType
    amount: Decimal
    status: OperationStatus
    balance: Decimal
    date_time: datetime = field(
        default_factory=lambda: datetime.now(tz=UTC),
    )

    def __str__(self) -> str:
        """Возвращает человекочитаемое описание операции."""
        return (
            f'тип операции: {self.type.value}, '
            f'сумма: {self.amount}, '
            f'статус: {self.status.value}, '
            f'баланс: {self.balance}, '
            f'время проведения: '
            f'{self.date_time.strftime("%Y-%m-%d %H:%M:%S")}'
        )

    def to_dict(self) -> dict:
        """Возвращает словарь для сериализации операции в json/csv."""
        return {
            'type': self.type.value,
            'amount': str(self.amount),
            'status': self.status.value,
            'balance': str(self.balance),
            'date_time': self.date_time.strftime('%Y-%m-%d %H:%M:%S'),
        }

In [5]:
class Account:
    """Банковский счёт: баланс, история операций и аналитика.

    Attributes:
        holder: Имя владельца счёта.
        account_number: Номер счёта в формате ACC-XXXX.
    """

    _account_counter = 1000
    account_type: AccountType | None = None
    ALLOWED_OPERATIONS = frozenset({'deposit', 'withdraw'})

    def __init__(
        self,
        account_holder: str,
        balance: Decimal = Decimal(0),
        *,
        account_number: str | None = None,
    ) -> None:
        """Создаёт счёт с проверкой имени владельца и баланса.

        Args:
            account_holder: Имя владельца в формате «Имя Фамилия».
            balance: Начальный баланс, не отрицательный.
            account_number: Явный номер счёта, иначе номер
                генерируется по счётчику классов.

        Raises:
            ValueError: Если имя владельца или баланс некорректны.
        """
        self.holder = account_holder
        self._operations_history: list[Operation] = []

        balance = self._to_decimal(balance)
        if balance < 0:
            raise ValueError('Баланс не может быть отрицательным')
        self._balance = balance

        if account_number is None:
            self._account_number = f'ACC-{Account._account_counter}'
            Account._account_counter += 1
        else:
            self._account_number = account_number

    def __repr__(self) -> str:
        """Возвращает представление счёта для отладки."""
        return (
            f'{type(self).__name__}(holder={self._holder!r}, '
            f'account_number={self._account_number!r}, '
            f'balance={self._balance})'
        )

    @property
    def holder(self) -> str:
        """Возвращает имя владельца счёта."""
        return self._holder

    @holder.setter
    def holder(self, value: str) -> None:
        """Проверяет формат имени владельца и сохраняет его.

        Args:
            value: Имя в формате «Имя Фамилия», кириллицей или
                латиницей, с заглавных букв.

        Raises:
            ValueError: Если формат имени нарушен.
        """
        full_name = value.strip() if isinstance(value, str) else ''
        if not HOLDER_PATTERN.match(full_name):
            raise ValueError(
                'Имя владельца должно быть в формате «Имя Фамилия»: '
                'с заглавных букв, кириллицей или латиницей.'
            )
        self._holder = full_name

    @property
    def account_number(self) -> str:
        """Возвращает номер счёта."""
        return self._account_number

    @staticmethod
    def _to_decimal(value: object) -> Decimal:
        """Приводит значение к типу Decimal.

        Args:
            value: Число или строка с числом.

        Returns:
            Значение в виде Decimal.

        Raises:
            ValueError: Если значение не является числом.
        """
        if isinstance(value, Decimal):
            return value
        try:
            return Decimal(str(value))
        except InvalidOperation as error:
            raise ValueError(f'Некорректная сумма: {value!r}') from error

    def _record_operation(
        self,
        operation_type: OperationType,
        amount: Decimal,
        status: OperationStatus,
    ) -> Operation:
        """Добавляет запись об операции в историю.

        Args:
            operation_type: Тип операции.
            amount: Сумма операции.
            status: Статус операции.

        Returns:
            Созданная запись об операции.
        """
        operation = Operation(
            type=operation_type,
            amount=amount,
            status=status,
            balance=self._balance,
        )
        self._operations_history.append(operation)
        return operation

    def deposit(self, amount: Decimal) -> None:
        """Пополняет счёт на указанную сумму.

        Args:
            amount: Сумма пополнения, положительная.

        Raises:
            ValueError: Если сумма неположительная.
        """
        amount = self._to_decimal(amount)
        if amount <= 0:
            self._record_operation(
                OperationType.DEPOSIT, amount, OperationStatus.FAIL,
            )
            raise ValueError('Сумма пополнения должна быть положительной.')
        self._balance = self._balance + amount
        self._record_operation(
            OperationType.DEPOSIT, amount, OperationStatus.SUCCESS,
        )

    def withdraw(self, amount: Decimal) -> bool:
        """Снимает средства со счёта.

        Args:
            amount: Сумма снятия, положительная.

        Returns:
            True при успехе, False при недостатке средств.

        Raises:
            ValueError: Если сумма неположительная.
        """
        amount = self._to_decimal(amount)
        if amount <= 0:
            self._record_operation(
                OperationType.WITHDRAW, amount, OperationStatus.FAIL,
            )
            raise ValueError('Сумма снятия должна быть положительной.')
        if amount > self._balance:
            self._record_operation(
                OperationType.WITHDRAW, amount, OperationStatus.FAIL,
            )
            return False
        self._balance = self._balance - amount
        self._record_operation(
            OperationType.WITHDRAW, amount, OperationStatus.SUCCESS,
        )
        return True

    def get_balance(self) -> Decimal:
        """Возвращает текущий баланс счёта."""
        return self._balance

    def operations_count(self) -> int:
        """Возвращает количество операций в истории."""
        return len(self._operations_history)

    def get_history(
        self,
        *,
        to_json: str | None = None,
        to_csv: str | None = None,
    ) -> str | None:
        """Возвращает историю операций в читаемом виде.

        Args:
            to_json: Путь для сохранения истории в json.
            to_csv: Путь для сохранения истории в csv.

        Returns:
            Текст истории либо None при сохранении в файл.

        Raises:
            ValueError: Если указаны оба формата сохранения.
        """
        return self._render_history(
            self._operations_history, to_json=to_json, to_csv=to_csv,
        )

    def get_largest_operations(
        self,
        n: int,
        *,
        to_json: str | None = None,
        to_csv: str | None = None,
    ) -> str | None:
        """Возвращает последние n крупных успешных операций.

        Отбирает успешные операции, сортирует их по убыванию суммы
        (при равенстве сумм — по убыванию даты) и берёт первые n.

        Args:
            n: Количество операций в выдаче.
            to_json: Путь для сохранения отбора в json.
            to_csv: Путь для сохранения отбора в csv.

        Returns:
            Текст отбора либо None при сохранении в файл.

        Raises:
            ValueError: Если указаны оба формата сохранения.
        """
        successful = [
            operation for operation in self._operations_history
            if operation.status is OperationStatus.SUCCESS
        ]
        largest = sorted(
            successful,
            key=lambda op: (op.amount, op.date_time),
            reverse=True,
        )[:n]
        return self._render_history(largest, to_json=to_json, to_csv=to_csv)

    @classmethod
    def _render_history(
        cls,
        operations: list[Operation],
        *,
        to_json: str | None = None,
        to_csv: str | None = None,
    ) -> str | None:
        """Формирует текст истории или сохраняет её в файл.

        Args:
            operations: Операции для вывода.
            to_json: Путь для сохранения истории в json.
            to_csv: Путь для сохранения истории в csv.

        Returns:
            Текст истории либо None при сохранении в файл.

        Raises:
            ValueError: Если указаны оба формата сохранения.
        """
        if to_json and to_csv:
            raise ValueError('Выберите один формат: to_json или to_csv.')
        rows = [operation.to_dict() for operation in operations]
        if to_json:
            with open(to_json, 'w', encoding='utf-8') as file:
                json.dump(rows, file, ensure_ascii=False, indent=2)
            return None
        if to_csv:
            with open(to_csv, 'w', encoding='utf-8', newline='') as file:
                writer = csv.DictWriter(file, fieldnames=OUTPUT_COLUMNS)
                writer.writeheader()
                writer.writerows(rows)
            return None
        return '\n'.join(str(operation) for operation in operations)

    @staticmethod
    def _read_records(path: str | Path) -> list[dict]:
        """Читает записи транзакций из файла csv или json.

        Args:
            path: Путь к файлу с расширением .csv или .json.

        Returns:
            Список словарей с полями транзакций.

        Raises:
            ValueError: Если расширение файла не поддерживается.
        """
        path = str(path)
        if path.endswith('.csv'):
            with open(path, encoding='utf-8') as file:
                return list(csv.DictReader(file))
        if path.endswith('.json'):
            with open(path, encoding='utf-8') as file:
                data = json.load(file)
            return data if isinstance(data, list) else [data]
        raise ValueError('Поддерживаются только файлы .csv и .json.')

    @staticmethod
    def _parse_date(raw: object) -> datetime | None:
        """Разбирает строку даты в поддерживаемых форматах.

        Args:
            raw: Исходное значение даты из файла.

        Returns:
            Дата и время операции либо None, если формат неверен.
        """
        if not isinstance(raw, str):
            return None
        for fmt in DATE_FORMATS:
            try:
                return datetime.strptime(raw, fmt).replace(tzinfo=UTC)
            except ValueError:
                continue
        return None

    @classmethod
    def _parse_record(cls, record: dict) -> dict | None:
        """Проверяет одну запись транзакции на корректность.

        Args:
            record: Словарь с полями транзакции из файла.

        Returns:
            Разобранная транзакция либо None, если данные с ошибками.
        """
        date = cls._parse_date(record.get('date'))
        if date is None:
            return None
        operation = record.get('operation')
        if operation not in cls.ALLOWED_OPERATIONS:
            return None
        status = record.get('status')
        if status not in {'success', 'fail'}:
            return None
        try:
            amount = Decimal(str(record.get('amount')))
            balance_after = Decimal(str(record.get('balance_after')))
        except (InvalidOperation, TypeError):
            return None
        if amount <= 0:
            return None
        return {
            'account_number': record.get('account_number'),
            'account_type': record.get('account_type'),
            'date': date,
            'operation': operation,
            'amount': amount,
            'balance_after': balance_after,
            'status': status,
        }

    @classmethod
    def clean_history(cls, records: list[dict]) -> list[dict]:
        """Ищет ошибки в данных перед занесением транзакций в историю.

        Отбрасывает записи с опечатками, отрицательными суммами,
        неверными датами и статусами, убирает дубликаты и сортирует
        транзакции по дате.

        Args:
            records: Сырые записи транзакций из файла.

        Returns:
            Очищенный и отсортированный по дате список транзакций.
        """
        cleaned: list[dict] = []
        seen: set = set()
        for record in records:
            parsed = cls._parse_record(record)
            if parsed is None:
                continue
            key = (
                parsed['account_number'],
                parsed['account_type'],
                parsed['date'],
                parsed['operation'],
                parsed['amount'],
                parsed['balance_after'],
                parsed['status'],
            )
            if key in seen:
                continue
            seen.add(key)
            cleaned.append(parsed)
        cleaned.sort(key=lambda item: item['date'])
        return cleaned

    @classmethod
    def from_file(
        cls,
        path: str | Path,
        account_number: str,
        account_holder: str,
    ) -> 'Account':
        """Загружает счёт с историей транзакций из файла csv или json.

        Отбирает записи своего счёта и своего типа, очищает их и
        восстанавливает баланс по последней операции.

        Args:
            path: Путь к файлу с транзакциями.
            account_number: Номер восстанавливаемого счёта.
            account_holder: Имя владельца счёта.

        Returns:
            Счёт с восстановленной историей операций и балансом.
        """
        records = cls._read_records(path)
        matching = [
            record for record in records
            if record.get('account_number') == account_number
            and record.get('account_type') == cls.account_type
        ]
        cleaned = cls.clean_history(matching)

        account = cls(
            account_holder,
            account_number=account_number,
        )
        account._operations_history = [
            Operation(
                type=OperationType(record['operation']),
                amount=record['amount'],
                status=OperationStatus(record['status']),
                balance=record['balance_after'],
                date_time=record['date'],
            )
            for record in cleaned
        ]
        if cleaned:
            account._balance = cleaned[-1]['balance_after']
        return account


In [6]:
class CheckingAccount(Account):
    """Расчётный счёт без дополнительных ограничений."""

    account_type = AccountType.CHECKING

In [7]:
class SavingsAccount(Account):
    """Сберегательный счёт: проценты на остаток и лимит снятия 50%."""

    account_type = AccountType.SAVINGS
    ALLOWED_OPERATIONS = frozenset({'deposit', 'withdraw', 'interest'})

    def apply_interest(self, rate: Decimal) -> Decimal:
        """Начисляет проценты на остаток и фиксирует операцию.

        Args:
            rate: Ставка процента, например 0.07 для 7% на остаток.

        Returns:
            Начисленная сумма процентов.

        Raises:
            ValueError: Если ставка отрицательная.
        """
        rate = self._to_decimal(rate)
        if rate < 0:
            raise ValueError('Ставка должна быть неотрицательной.')
        interest = self._balance * rate
        self._balance = self._balance + interest
        self._record_operation(
            OperationType.INTEREST, interest, OperationStatus.SUCCESS,
        )
        return interest

    def withdraw(self, amount: Decimal) -> bool:
        """Снимает средства, но не более 50% текущего баланса.

        Args:
            amount: Сумма снятия, положительная.

        Returns:
            True при успехе, False при превышении лимита снятия
            или недостатке средств.

        Raises:
            ValueError: Если сумма неположительная.
        """
        amount = self._to_decimal(amount)
        if amount > 0 and amount > self._balance * Decimal('0.5'):
            self._record_operation(
                OperationType.WITHDRAW, amount, OperationStatus.FAIL,
            )
            return False
        return super().withdraw(amount)

In [8]:
data_dir = "./"

print('== 1. Базовые операции ==')
account = Account('John Wick', Decimal(100))
print(f'создан: {account!r}')
account.deposit(Decimal(50))
print(f'deposit(50) -> {account.get_balance()}')
insuff = account.withdraw(Decimal(200))
print(f'withdraw(200) -> {insuff} (недостаточно средств)')
ok = account.withdraw(Decimal(30))
print(f'withdraw(30) -> {ok}, баланс: {account.get_balance()}')
try:
    account.deposit(Decimal(-10))
except ValueError as exc:
    print(f'deposit(-10) -> ValueError: {exc}')
print('история:')
print(account.get_history())

print('\n== 2. Валидация имени ==')
for name in ('john wick', 'John', 'John Wick'):
    try:
        Account(name)
        print(f'{name!r} -> ок')
    except ValueError:
        print(f'{name!r} -> ValueError: неверный формат имени')

print('\n== 3. CheckingAccount / SavingsAccount ==')
checking = CheckingAccount('Jane Doe', Decimal(1000))
savings = SavingsAccount('Джон Уик', Decimal(1000))
print(f'создан: {checking!r}')
print(f'создан: {savings!r}')
print(f'типы: {checking.account_type}, {savings.account_type}')

interest = savings.apply_interest(Decimal('0.07'))
print(f'apply_interest(7%) -> {interest}, '
      f'баланс: {savings.get_balance()}')
over = savings.withdraw(Decimal(600))
print(f'Savings withdraw(600) >50% -> {over}')
half = savings.withdraw(Decimal(500))
print(f'Savings withdraw(500) <=50% -> {half}, '
      f'баланс: {savings.get_balance()}')

print('\n== 4. Крупные операции ==')
print(savings.get_largest_operations(2))

print('\n== 5. Загрузка из файлов ==')
restored = CheckingAccount.from_file(
    './transactions_dirty.csv', 'ACC-100001', 'John Wick',
)
print(f'{restored.account_number}: операций '
      f'{restored.operations_count()}, баланс {restored.get_balance()}')
saved = SavingsAccount.from_file(
    './transactions_dirty.json', 'ACC-100002', 'Джон Уик',
)
print(f'{saved.account_number}: операций '
      f'{saved.operations_count()}, баланс {saved.get_balance()}')

print('\n== 6. Вывод истории в json/csv ==')
account.get_history(to_json='history_sample.json')
account.get_history(to_csv='history_sample.csv')
print('записано: history_sample.json, history_sample.csv')


== 1. Базовые операции ==
создан: Account(holder='John Wick', account_number='ACC-1000', balance=100)
deposit(50) -> 150
withdraw(200) -> False (недостаточно средств)
withdraw(30) -> True, баланс: 120
deposit(-10) -> ValueError: Сумма пополнения должна быть положительной.
история:
тип операции: deposit, сумма: 50, статус: success, баланс: 150, время проведения: 2026-10-04 14:36:13
тип операции: withdraw, сумма: 200, статус: fail, баланс: 150, время проведения: 2026-10-04 14:36:13
тип операции: withdraw, сумма: 30, статус: success, баланс: 120, время проведения: 2026-10-04 14:36:13
тип операции: deposit, сумма: -10, статус: fail, баланс: 120, время проведения: 2026-10-04 14:36:13

== 2. Валидация имени ==
'john wick' -> ValueError: неверный формат имени
'John' -> ValueError: неверный формат имени
'John Wick' -> ок

== 3. CheckingAccount / SavingsAccount ==
создан: CheckingAccount(holder='Jane Doe', account_number='ACC-1002', balance=1000)
создан: SavingsAccount(holder='Джон Уик', accoun